# Notebook 01b — Country Threshold & Balancing

Takes the unfiltered output of NB01 v3 (`articles_v3_all.csv`) and produces a balanced dataset for downstream analysis.

**Two independent knobs:**
- `MIN_ARTICLES_PER_COUNTRY` — minimum articles for a country to be included
- `BALANCE_TARGET_N` — articles per country after downsampling

This notebook is cheap to rerun with different parameters — no WARC reprocessing needed.

---

## 0. Setup: mount Drive, imports, configuration

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from pathlib import Path

# --- Configurable knobs ---
MIN_ARTICLES_PER_COUNTRY = 100      # inclusion threshold — adjust after seeing the distribution
BALANCE_TARGET_N         = None     # None = auto (= smallest included country); or set int e.g. 150
RANDOM_SEED              = 42

# --- Paths ---
DRIVE_BASE = Path('/content/drive/MyDrive/UNI/JLU M.Sc Data Analytics/'
                  'Semester 3/Spezialisierungmodule/data')
INPUT_FILE  = DRIVE_BASE / 'processed' / 'articles_with_country_2025_v3.csv'
OUTPUT_FILE = DRIVE_BASE / 'processed' / 'articles_balanced_2025_v3.csv'

print(f'Input file : {INPUT_FILE}')
print(f'Output file: {OUTPUT_FILE}')
print(f'Exists?    : {INPUT_FILE.exists()}')

Input file : /content/drive/MyDrive/UNI/JLU M.Sc Data Analytics/Semester 3/Spezialisierungmodule/data/processed/articles_with_country_2025_v3.csv
Output file: /content/drive/MyDrive/UNI/JLU M.Sc Data Analytics/Semester 3/Spezialisierungmodule/data/processed/articles_balanced_2025_v3.csv
Exists?    : True


---
## 1. Load unfiltered data from NB01 v3

No filtering yet — just look at the full country distribution so we can make an informed choice about the threshold.

In [3]:
df = pd.read_csv(INPUT_FILE)
print(f'Total articles loaded: {len(df):,}')
print(f'Distinct countries:    {df["country_url"].nunique()}')

dist = (df['country_url']
        .value_counts()
        .rename_axis('country')
        .reset_index(name='n_articles'))

print('\n=== Country distribution (raw, before threshold) ===')
print(dist.to_string(index=False))

Total articles loaded: 67,870
Distinct countries:    22

=== Country distribution (raw, before threshold) ===
country  n_articles
     IN        8719
     US        5763
     UK         565
     AE         499
     NG         334
     PK         312
     CN         236
     AU         231
     SG         158
     ZA         134
     KR         106
     KE         102
     IL          94
     MY          91
     QA          89
     ZM          38
     DE          37
     FJ          37
     AT          36
     TH          27
     IR          25
     CA          10


---
## 2. Apply threshold and balance

**Step 1:** Drop countries with fewer than `MIN_ARTICLES_PER_COUNTRY` articles.  
**Step 2:** Downsample every surviving country to `BALANCE_TARGET_N` articles (or to the smallest included count if `None`).

Both steps use `RANDOM_SEED = 42` for reproducibility.

In [4]:
# --- Step 1: threshold ---
included = dist.loc[dist['n_articles'] >= MIN_ARTICLES_PER_COUNTRY, 'country'].tolist()
excluded = dist.loc[dist['n_articles'] <  MIN_ARTICLES_PER_COUNTRY, 'country'].tolist()

print(f'Included ({len(included)}): {included}')
print(f'Excluded ({len(excluded)}): {excluded}')

df_included = df[df['country_url'].isin(included)].copy()

# --- Step 2: balance ---
smallest = df_included['country_url'].value_counts().min()
target_n = BALANCE_TARGET_N if BALANCE_TARGET_N is not None else smallest

if BALANCE_TARGET_N is not None and BALANCE_TARGET_N > smallest:
    raise ValueError(
        f'BALANCE_TARGET_N={BALANCE_TARGET_N} exceeds smallest included country '
        f'({smallest}). Lower the target or raise MIN_ARTICLES_PER_COUNTRY.'
    )

df_balanced = (df_included
               .groupby('country_url', group_keys=False)
               .apply(lambda g: g.sample(n=target_n, random_state=RANDOM_SEED))
               .reset_index(drop=True))

print(f'\nBalanced dataset: {len(df_balanced):,} articles '
      f'({len(included)} countries × {target_n} articles)')

Included (12): ['IN', 'US', 'UK', 'AE', 'NG', 'PK', 'CN', 'AU', 'SG', 'ZA', 'KR', 'KE']
Excluded (10): ['IL', 'MY', 'QA', 'ZM', 'DE', 'FJ', 'AT', 'TH', 'IR', 'CA']

Balanced dataset: 1,224 articles (12 countries × 102 articles)


/tmp/ipykernel_8836/1258047519.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=target_n, random_state=RANDOM_SEED))


---
## 3. Save balanced dataset to Drive

In [5]:
df_balanced.to_csv(OUTPUT_FILE, index=False)
print(f'Saved → {OUTPUT_FILE.name}')
print(f'Rows  : {len(df_balanced):,}')

# Confirm final balance
print('\n=== Final balanced distribution ===')
print(df_balanced['country_url'].value_counts().to_string())

Saved → articles_balanced_2025_v3.csv
Rows  : 1,224

=== Final balanced distribution ===
country_url
AE    102
AU    102
CN    102
IN    102
KE    102
KR    102
NG    102
PK    102
SG    102
UK    102
US    102
ZA    102
